## Get file list from GitHub API

In [0]:
import requests

repo_api = "https://api.github.com/repos/JeffSackmann/tennis_atp/contents"
response = requests.get(repo_api)
files = response.json()

csv_files = [
    f for f in files
    if f["name"].endswith(".csv")
]

len(csv_files)

168

## Sample item

In [0]:
csv_files[0]

{'name': 'atp_matches_1968.csv',
 'path': 'atp_matches_1968.csv',
 'sha': '336491b9aaf99b12972a40f97c3bbd792fb86745',
 'size': 673978,
 'url': 'https://api.github.com/repos/JeffSackmann/tennis_atp/contents/atp_matches_1968.csv?ref=master',
 'html_url': 'https://github.com/JeffSackmann/tennis_atp/blob/master/atp_matches_1968.csv',
 'git_url': 'https://api.github.com/repos/JeffSackmann/tennis_atp/git/blobs/336491b9aaf99b12972a40f97c3bbd792fb86745',
 'download_url': 'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_1968.csv',
 'type': 'file',
 '_links': {'self': 'https://api.github.com/repos/JeffSackmann/tennis_atp/contents/atp_matches_1968.csv?ref=master',
  'git': 'https://api.github.com/repos/JeffSackmann/tennis_atp/git/blobs/336491b9aaf99b12972a40f97c3bbd792fb86745',
  'html': 'https://github.com/JeffSackmann/tennis_atp/blob/master/atp_matches_1968.csv'}}

## Download each CSV into ADLS

In [0]:
bronze_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/bronze"

for f in csv_files:
    raw_url = f["download_url"]
    file_name = f["name"]

    r = requests.get(raw_url)
    r.raise_for_status()

    dbutils.fs.put(
        f"{bronze_path}/{file_name}",
        r.text,
        overwrite=True
    )

Wrote 673978 bytes.
Wrote 494788 bytes.
Wrote 512410 bytes.
Wrote 566131 bytes.
Wrote 563440 bytes.
Wrote 676606 bytes.
Wrote 671158 bytes.
Wrote 664790 bytes.
Wrote 623651 bytes.
Wrote 658101 bytes.
Wrote 614875 bytes.
Wrote 632805 bytes.
Wrote 640025 bytes.
Wrote 625944 bytes.
Wrote 655284 bytes.
Wrote 563037 bytes.
Wrote 531124 bytes.
Wrote 558319 bytes.
Wrote 534012 bytes.
Wrote 583360 bytes.
Wrote 615887 bytes.
Wrote 592902 bytes.
Wrote 630930 bytes.
Wrote 743040 bytes.
Wrote 760047 bytes.
Wrote 779735 bytes.
Wrote 787913 bytes.
Wrote 762360 bytes.
Wrote 757247 bytes.
Wrote 730300 bytes.
Wrote 725069 bytes.
Wrote 673043 bytes.
Wrote 680357 bytes.
Wrote 669112 bytes.
Wrote 649377 bytes.
Wrote 649296 bytes.
Wrote 662757 bytes.
Wrote 659963 bytes.
Wrote 660698 bytes.
Wrote 644456 bytes.
Wrote 631452 bytes.
Wrote 626700 bytes.
Wrote 614299 bytes.
Wrote 611939 bytes.
Wrote 610116 bytes.
Wrote 596714 bytes.
Wrote 589216 bytes.
Wrote 594438 bytes.
Wrote 617660 bytes.
Wrote 611380 bytes.


## Auto-organize files

In [0]:
import re

base_path = "abfss://<container>@<storage-account>.dfs.core.windows.net/<env>/<user>/<project>/bronze"

paths = {
    "players": f"{base_path}/players/",
    "rankings": f"{base_path}/rankings/",
    "singles": f"{base_path}/matches/singles/",
    "doubles": f"{base_path}/matches/doubles/",
    "challengers": f"{base_path}/matches/challengers_qual/",
    "futures": f"{base_path}/matches/futures/",
    "amateur": f"{base_path}/matches/amateur/",
    "other": f"{base_path}/matches/other/"
}

# Create directories
for p in paths.values():
    dbutils.fs.mkdirs(p)

files = dbutils.fs.ls(base_path)

for f in files:
    if f.isDir():
        continue

    name = f.name.lower()

    # Players
    if name.startswith("atp_players"):
        target = paths["players"]

    # Rankings
    elif name.startswith("atp_rankings"):
        target = paths["rankings"]

    # Doubles
    elif name.startswith("atp_matches_doubles"):
        target = paths["doubles"]

    # Challengers / Qualifiers
    elif name.startswith("atp_matches_qual_chall"):
        target = paths["challengers"]

    # Futures
    elif name.startswith("atp_matches_futures"):
        target = paths["futures"]

    # Amateur
    elif name == "atp_matches_amateur.csv":
        target = paths["amateur"]

    # Singles ONLY if year-based
    elif re.match(r"atp_matches_\d{4}\.csv", name):
        target = paths["singles"]

    # Everything else
    else:
        target = paths["other"]

    dbutils.fs.mv(f.path, target + f.name)

## Displaying the key paths

In [0]:
from pyspark.sql.functions import col, from_unixtime

def ls_no_path_formatted(p):
    df = spark.createDataFrame(dbutils.fs.ls(p))
    return (
        df.drop("path")
          .withColumn(
              "modificationTime",
              from_unixtime(col("modificationTime") / 1000)
          )
    )

display(ls_no_path_formatted(f"{base_path}/matches"))
display(ls_no_path_formatted(f"{base_path}/matches/singles"))

name,size,modificationTime
amateur/,0,2025-12-31 14:08:51
challengers_qual/,0,2025-12-31 14:08:51
doubles/,0,2025-12-31 14:08:51
futures/,0,2025-12-31 14:08:51
other/,0,2025-12-31 14:08:51
singles/,0,2025-12-31 14:08:51


name,size,modificationTime
atp_matches_1968.csv,673978,2026-01-09 11:53:15
atp_matches_1969.csv,494788,2026-01-09 11:53:16
atp_matches_1970.csv,512410,2026-01-09 11:53:16
atp_matches_1971.csv,566131,2026-01-09 11:53:16
atp_matches_1972.csv,563440,2026-01-09 11:53:17
atp_matches_1973.csv,676606,2026-01-09 11:53:17
atp_matches_1974.csv,671158,2026-01-09 11:53:17
atp_matches_1975.csv,664790,2026-01-09 11:53:17
atp_matches_1976.csv,623651,2026-01-09 11:53:18
atp_matches_1977.csv,658101,2026-01-09 11:53:18
